# Đánh giá hiệu suất mô hình bằng chỉ số BLEU

Tệp notebook này thực hiện đánh giá mô hình LLaMA 3.1 8B trước và sau khi fine-tune trên tập dữ liệu kiểm thử (test dataset) bằng chỉ số BLEU.


In [1]:
!pip install -U langchain langchain-community qdrant-client sentence-transformers transformers accelerate bitsandbytes huggingface_hub evaluate sacrebleu tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 106.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 25.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.1/512.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━

In [2]:
import os
import torch
import json 
from tqdm import tqdm # Theo dõi tiến trình

from langchain_community.vectorstores import Qdrant
from langchain_community.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from huggingface_hub import login
import evaluate # Cho các metrics

2025-06-09 03:30:45.573960: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749439845.972210      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749439846.088914      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("1HF_TOKEN")
login(token=hf_token)

In [4]:
def generate_answer_from_pipeline(text_pipeline, prompt_text: str, current_tokenizer):
    if not text_pipeline:
        return "LỖI: Pipeline chưa được khởi tạo."
    terminators = [
        current_tokenizer.eos_token_id,
        current_tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]
    terminators = [term for term in terminators if term is not None]
    try:
        outputs = text_pipeline(
            prompt_text, max_new_tokens=512, do_sample=True, temperature=0.6,
            top_p=0.9, pad_token_id=current_tokenizer.pad_token_id,
            eos_token_id=terminators if terminators else current_tokenizer.eos_token_id
        )
        generated_text_with_prompt = outputs[0]['generated_text']
        answer = generated_text_with_prompt
        if generated_text_with_prompt.startswith(prompt_text):
            answer = generated_text_with_prompt[len(prompt_text):].strip()
        else:
            assistant_signal_llama3 = "<|start_header_id|>assistant<|end_header_id|>\n\n"
            idx_signal = generated_text_with_prompt.rfind(assistant_signal_llama3)
            if idx_signal != -1:
                answer = generated_text_with_prompt[idx_signal + len(assistant_signal_llama3):].strip()
        for term_token_str in [current_tokenizer.eos_token, "<|eot_id|>"]:
            if term_token_str and answer.endswith(term_token_str):
                answer = answer[:-len(term_token_str)].strip()
        return answer
    except Exception as e:
        print(f"Lỗi trong quá trình generate text: {e}")
        return "Lỗi sinh câu trả lời."

In [5]:
def format_prompt_finetuned_for_eval(context: str, question: str, tokenizer_ft):
    system_prompt = "Bạn là một chuyên gia về y tế và sức khỏe. Hãy sử dụng ngữ cảnh được cung cấp để trả lời câu hỏi."
    user_message = f"Ngữ cảnh: {context}\nCâu hỏi: {question}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    return tokenizer_ft.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [6]:
HF_MODEL_REPO_ID_FINETUNED = "tralalelo/fine-tune-Llama3.1-8B-merged-4bit-28-5"
tokenizer_finetuned, model_finetuned, text_generator_finetuned = None, None, None
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
print(f"\n--- Đang tải mô hình FINE-TUNE: {HF_MODEL_REPO_ID_FINETUNED} ---")
try:
    tokenizer_finetuned = AutoTokenizer.from_pretrained(HF_MODEL_REPO_ID_FINETUNED, trust_remote_code=True)
    if tokenizer_finetuned.pad_token is None and tokenizer_finetuned.eos_token:
        tokenizer_finetuned.pad_token = tokenizer_finetuned.eos_token
    model_finetuned = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_REPO_ID_FINETUNED, quantization_config=quantization_config ,
        device_map="auto", trust_remote_code=True
    )
    model_finetuned.eval()
    text_generator_finetuned = pipeline("text-generation", model=model_finetuned, tokenizer=tokenizer_finetuned)
    print("Mô hình fine-tune đã tải thành công.")
except Exception as e:
    print(f"LỖI khi tải mô hình fine-tune: {e}")



--- Đang tải mô hình FINE-TUNE: tralalelo/fine-tune-Llama3.1-8B-merged-4bit-28-5 ---


tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Device set to use cuda:0


Mô hình fine-tune đã tải thành công.


In [7]:
TEST_DATA_FILE_PATH = "/kaggle/input/test-data/test.jsonl"

test_dataset = []
try:
    with open(TEST_DATA_FILE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                test_dataset.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Bỏ qua dòng không hợp lệ: {line.strip()}")
    print(f"Đã tải {len(test_dataset)} mẫu từ tập test: {TEST_DATA_FILE_PATH}")
    if not test_dataset:
        raise ValueError("Tập test rỗng hoặc không tải được.")
except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file test data tại '{TEST_DATA_FILE_PATH}'. Vui lòng cung cấp đường dẫn đúng.")
    test_dataset = []
except Exception as e:
    print(f"Lỗi khi tải test data: {e}")
    test_dataset = []

Đã tải 750 mẫu từ tập test: /kaggle/input/test-data/test.jsonl


In [8]:
# Trích xuất references
references = [item.get('answer', '') for item in test_dataset if isinstance(item, dict)]
if not all(isinstance(ref, str) for ref in references):
    print("CẢNH BÁO: Một số 'answer' trong tập test không phải là string hoặc bị thiếu.")
    references = [str(ref) if ref is not None else "" for ref in references] # Đảm bảo là list of strings

In [9]:
# SINH CÁC CÂU TRẢ LỜI DỰ ĐOÁN VÀ LƯU FILE
predictions_finetuned_list = []
predictions_base_list = []

OUTPUT_FILE_FINETUNED = "predictions_finetuned.jsonl"
OUTPUT_FILE_BASE = "predictions_base.jsonl"

print("\nBắt đầu sinh câu trả lời trên tập test và lưu kết quả...")

# Model Fine-tuned
if text_generator_finetuned and tokenizer_finetuned and test_dataset:
    print(f"\nĐang xử lý với mô hình Fine-tuned... Kết quả sẽ được lưu vào: {OUTPUT_FILE_FINETUNED}")
    with open(OUTPUT_FILE_FINETUNED, 'w', encoding='utf-8') as outfile_ft:
        for item in tqdm(test_dataset, desc="Fine-tuned Model Predictions"):
            question = item.get('question', '')
            context_original = item.get('context', '') 
            reference_answer = item.get('answer', '') 

            prompt_ft = format_prompt_finetuned_for_eval(context_original, question, tokenizer_finetuned)
            answer_ft = generate_answer_from_pipeline(text_generator_finetuned, prompt_ft, tokenizer_finetuned)
            predictions_finetuned_list.append(answer_ft)

            # Tạo một dictionary chứa thông tin cần lưu
            result_item = {
                "question": question,
                "original_context": context_original, # Lưu ngữ cảnh gốc
                "reference_answer": reference_answer,
                "predicted_answer_finetuned": answer_ft
            }
            # Ghi JSON object vào file, mỗi object một dòng
            outfile_ft.write(json.dumps(result_item, ensure_ascii=False) + '\n')
    print(f"Đã lưu kết quả dự đoán của mô hình Fine-tuned vào {OUTPUT_FILE_FINETUNED}")
else:
    print("Bỏ qua sinh dự đoán cho mô hình fine-tune (model/tokenizer/dataset không sẵn sàng).")


Bắt đầu sinh câu trả lời trên tập test và lưu kết quả...

Đang xử lý với mô hình Fine-tuned... Kết quả sẽ được lưu vào: predictions_finetuned.jsonl


Fine-tuned Model Predictions: 100%|██████████| 750/750 [4:54:57<00:00, 23.60s/it]  

Đã lưu kết quả dự đoán của mô hình Fine-tuned vào predictions_finetuned.jsonl


In [12]:
def format_prompt_base_llama3_instruct_for_eval(context: str, question: str, tokenizer_b):
    system_prompt = "Bạn là một chuyên gia về y tế và sức khỏe. Hãy sử dụng ngữ cảnh được cung cấp để trả lời câu hỏi."
    user_prompt_with_context = f"Ngữ cảnh: {context}\nCâu hỏi: {question}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_with_context},
    ]
    return tokenizer_b.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [14]:
HF_MODEL_REPO_ID_BASE = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer_base, model_base, text_generator_base = None, None, None
print(f"\n--- Đang tải mô hình GỐC: {HF_MODEL_REPO_ID_BASE} ---")
try:
    tokenizer_base = AutoTokenizer.from_pretrained(HF_MODEL_REPO_ID_BASE, token=hf_token, trust_remote_code=True)
    if tokenizer_base.pad_token is None and tokenizer_base.eos_token:
        tokenizer_base.pad_token = tokenizer_base.eos_token
    model_base = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_REPO_ID_BASE, quantization_config=quantization_config,
        device_map="auto", token=hf_token, trust_remote_code=True
    )
    model_base.eval()
    text_generator_base = pipeline("text-generation", model=model_base, tokenizer=tokenizer_base)
    print("Mô hình gốc đã tải thành công.")
except Exception as e:
    print(f"LỖI khi tải mô hình gốc: {e}")


--- Đang tải mô hình GỐC: meta-llama/Meta-Llama-3.1-8B-Instruct ---


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Device set to use cuda:0


Mô hình gốc đã tải thành công.


In [16]:
# Model Gốc
if text_generator_base and tokenizer_base and test_dataset:
    print(f"\nĐang xử lý với mô hình Base... Kết quả sẽ được lưu vào: {OUTPUT_FILE_BASE}")
    with open(OUTPUT_FILE_BASE, 'w', encoding='utf-8') as outfile_base:
        for item in tqdm(test_dataset, desc="Base Model Predictions"):
            question = item.get('question', '')
            context_original = item.get('context', '')
            reference_answer = item.get('answer', '')
            context_for_prompt = context_original

            prompt_b = format_prompt_base_llama3_instruct_for_eval(context_for_prompt, question, tokenizer_base)
            answer_b = generate_answer_from_pipeline(text_generator_base, prompt_b, tokenizer_base)
            predictions_base_list.append(answer_b)

            result_item = {
                "question": question,
                "original_context": context_original,
                "context_used_for_prompt": context_for_prompt,
                "reference_answer": reference_answer,
                "predicted_answer_base": answer_b
            }
            outfile_base.write(json.dumps(result_item, ensure_ascii=False) + '\n')
    print(f"Đã lưu kết quả dự đoán của mô hình Base vào {OUTPUT_FILE_BASE}")
else:
    print("Bỏ qua sinh dự đoán cho mô hình base (model/tokenizer/dataset không sẵn sàng).")


Đang xử lý với mô hình Base... Kết quả sẽ được lưu vào: predictions_base.jsonl


Base Model Predictions: 100%|██████████| 750/750 [5:34:13<00:00, 26.74s/it]  

Đã lưu kết quả dự đoán của mô hình Base vào predictions_base.jsonl


In [19]:
bleu_metric = evaluate.load('bleu')

def compute_bleu_for_model(model_name, predictions, references_list_str):
    """
    Tính toán và in điểm BLEU cho một mô hình.
    Args:
        model_name (str): Tên của mô hình.
        predictions (list of str): Danh sách các câu trả lời dự đoán.
        references_list_str (list of str): Danh sách các câu trả lời tham chiếu.
    """
    global bleu_metric 

    if bleu_metric is None:
        try:
            print("Đang thử tải lại metric BLEU...")
            bleu_metric = evaluate.load('bleu')
            print("Metric BLEU đã được tải thành công.")
        except Exception as e:
            print(f"Lỗi khi tải metric BLEU: {e}")
            print("Hãy đảm bảo bạn đã cài đặt: pip install evaluate sacrebleu")
            return # Không thể tiếp tục nếu không có metric

    if not predictions:
        print(f"Không có predictions cho mô hình {model_name} để đánh giá BLEU.")
        return
    if not references_list_str:
        print(f"Không có references cho mô hình {model_name} để đánh giá BLEU.")
        return
    if len(predictions) != len(references_list_str):
        print(f"LỖI: Số lượng predictions ({len(predictions)}) và references ({len(references_list_str)}) không khớp cho model {model_name}.")
        return

    print(f"\n--- Tính điểm BLEU cho mô hình: {model_name} ---")
    try:
        bleu_results = bleu_metric.compute(predictions=predictions, references=references_list_str)

        print(f"  BLEU Score: {bleu_results['bleu'] * 100:.2f}")
        print(f"  Precisions (1-gram to 4-gram):")
        for i, prec in enumerate(bleu_results['precisions']):
            print(f"    {i+1}-gram: {prec * 100:.2f}")
        print(f"  Brevity Penalty: {bleu_results['brevity_penalty']:.4f}")
        print(f"  Length Ratio: {bleu_results['length_ratio']:.4f}")
    
    except Exception as e:
        print(f"Lỗi khi tính BLEU cho mô hình {model_name}: {e}")
        import traceback
        traceback.print_exc()

# Chạy đánh giá BLEU cho mô hình Fine-tuned

if 'references' in locals() and references:
    if 'predictions_finetuned_list' in locals() and predictions_finetuned_list:
        # Kiểm tra lại số lượng phần tử khớp nhau
        if len(predictions_finetuned_list) == len(references):
            compute_bleu_for_model("Base Model", predictions_finetuned_list, references)
        else:
            print(f"Số lượng predictions_finetuned ({len(predictions_finetuned_list)}) không khớp với references ({len(references)}). Không thể tính BLEU.")
    else:
        print("Không có `predictions_finetuned_list` để đánh giá BLEU. "
              "Có thể bạn cần tải lại từ file nếu đã chạy trước đó và không lưu vào biến.")
       
    if 'predictions_base_list' in locals() and predictions_base_list:
        if len(predictions_base_list) == len(references):
            compute_bleu_for_model("Fine-tuned Model", predictions_base_list, references)
        else:
             print(f"Số lượng predictions_base ({len(predictions_base_list)}) không khớp với references ({len(references)}). Không thể tính BLEU.")
   
else:
    print("Không có dữ liệu tham chiếu (references) từ tập test để thực hiện đánh giá BLEU. "
          "Hãy đảm bảo `test_dataset` đã được tải và `references` đã được trích xuất.")

print("\n--- Hoàn thành đánh giá BLEU ---")


--- Tính điểm BLEU cho mô hình: Base Model ---
  BLEU Score: 3.62
  Precisions (1-gram to 4-gram):
    1-gram: 18.39
    2-gram: 5.43
    3-gram: 1.97
    4-gram: 0.88
  Brevity Penalty: 1.0000
  Length Ratio: 2.0116

--- Tính điểm BLEU cho mô hình: Fine-tuned Model ---
  BLEU Score: 3.72
  Precisions (1-gram to 4-gram):
    1-gram: 18.70
    2-gram: 5.55
    3-gram: 2.01
    4-gram: 0.91
  Brevity Penalty: 1.0000
  Length Ratio: 1.9811

--- Hoàn thành đánh giá BLEU ---
